In [156]:
import warnings
from itertools import groupby

warnings.filterwarnings("ignore")

In [157]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns

In [158]:
from sklearn.preprocessing import MinMaxScaler

## 1. Configuration

In [159]:
RANDOM_STATE = 42

PROCESSED_DIR = Path("../artifacts/processed")
FEATURE_DIR = Path("../artifacts/features")

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

Top_k = 10

RECENCY_LEMBDA = 0.05

## 2. Load Processed Datasets

In [160]:
consumer = pd.read_parquet(PROCESSED_DIR / "consumer_clean.parquet")
print("Consumer Shape", consumer.shape)

content = pd.read_parquet(PROCESSED_DIR / "content_clean.parquet")
print("Content Shape", content.shape)

Consumer Shape (72312, 11)
Content Shape (3122, 16)


## 3. Standardize Column Names

In [161]:
consumer.columns = (consumer.columns.str.strip().str.lower())

content.columns = (content.columns.str.strip().str.lower())

## 4. Convert TimeStamps

- Convert Unix timestamp into timezone aware datetime values
- Consumer timestamp : when the user interacted with content
- Content timestamp : content lifecycle / event timestep

In [162]:
consumer["event_datetime"] = pd.to_datetime(consumer["event_timestamp"],unit="s",utc=True)

content["content_event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True)

## 5. Consumer Temporal Features

In [163]:
# Hour of  day
consumer["event_hour"] = (consumer["event_datetime"].dt.hour)

# Day of week.
consumer["event_day_of_week"] = (consumer["event_datetime"].dt.dayofweek)

# Day name.
consumer["event_day_name"] = (consumer["event_datetime"].dt.day_name())

# Weekend indicator.
consumer["is_weekend"] = (consumer["event_day_of_week"] >= 5).astype(int)

# Month.
consumer["event_month"] = (consumer["event_datetime"].dt.month)

# Week of year.
consumer["event_week"] = (consumer["event_datetime"].dt.isocalendar().week.astype(int))

# Date-level activity.
consumer["event_date"] = (consumer["event_datetime"].dt.date)

## 6. Create Interaction Strength Features

- One of the most important features for ALS

In [164]:
# Initial implicit-feedback weights These are modeling assumptions and should later be tuned using validation experiments.
INTERACTION_WEIGHTS = {"content_watched": 1.0, "content_liked": 2.0, "content_saved": 3.0,"content_followed": 4.0,"content_commented_on": 5.0}

# Map interaction type to numerical strength.
consumer["interaction_strength"] = (consumer["interaction_type"].map(INTERACTION_WEIGHTS).fillna(0.0))

# Binary positive interaction indicator.
consumer["is_positive_interaction"] = (consumer["interaction_strength"] > 0).astype(int)

## 7. Interaction Type One Hot Features

In [165]:
interaction_types = ["content_watched","content_liked","content_saved","content_followed","content_commented_on"]

for interaction in interaction_types:
    feature_name = ("is_" + interaction.replace("content_", ""))

    consumer[feature_name] = (consumer["interaction_type"].eq(interaction).astype(int))

In [166]:
content["is_available"] = (content["interaction_type"].eq("content_present")).astype(int)

## 8. Article / Content Status

In [167]:
# determine which articles are currently represented as content present versus pulled out
content["is_available"] = (content["interaction_type"].eq("content_available")).astype(int)

## 9. Latest Aritcle Status

In [168]:
content_status = (content.sort_values(["item_id","content_event_datetime"]).groupby("item_id").tail(1).copy())

content_status = content_status[["item_id","content_event_datetime", "interaction_type","is_available"]].rename(columns={"interaction_type":"latest_content_status"})

## 10. Clean Article Catelog

In [169]:
article_columns = ["item_id","producer_id","item_type","title","text_description","language","item_url","producer_country","producer_location"]

article_features = (content[article_columns].drop_duplicates(subset=["item_id"]).copy())

## 11. Text Features

In [170]:
# Replace missing text with empty strings.
article_features["title"] = (article_features["title"].fillna("").astype(str))
article_features["text_description"] = (article_features["text_description"].fillna("").astype(str))

# Number of words in title.
article_features["title_word_count"] = (article_features["title"].str.split().str.len())

# Number of words in description.
article_features["description_word_count"] = (article_features["text_description"].str.split().str.len())

# Number of characters in title.
article_features["title_char_count"] = (article_features["title"].str.len())

# Number of characters in description.
article_features["description_char_count"] = (article_features["text_description"].str.len())

## 12. Article Text Quality Flags

In [171]:
article_features["has_title"] = (article_features["title_word_count"] > 0).astype(int)

article_features["has_description"] = (article_features["description_word_count"] > 0).astype(int)

article_features["has_sufficient_text"] = ((article_features["title_word_count"] >= 3) & (article_features["description_word_count"] >= 20)).astype(int)

## 13. English Language Feature

In [172]:
article_features["is_english"] = (article_features["language"].astype(str).str.lower().eq("en")).astype(int)

## 14. Content Type Features

In [173]:
article_features["is_video"] = (article_features["item_type"].astype(str).str.upper().eq("VIDEO")).astype(int)

article_features["is_html"] = (article_features["item_type"].astype(str).str.upper().eq("HTML")).astype(int)

article_features["is_rich"] = (article_features["item_type"].astype(str).str.upper().eq("RICH")).astype(int)

## 15. Article Interaction Aggregates

In [174]:
article_interactions = (consumer.groupby("item_id").agg(
        # Total number of events.
        article_total_interactions=("consumer_id", "size"),
        # Number of unique users.
        article_unique_users=("consumer_id","nunique"),
        # Number of unique sessions.
        article_unique_sessions=("consumer_session_id","nunique"),
        # Total implicit feedback strength.
        article_total_strength=("interaction_strength","sum"),
        # Average interaction strength.
        article_avg_strength=("interaction_strength","mean")).reset_index()
)

In [175]:
article_interactions

,item_id,article_total_interactions,article_unique_users,article_unique_sessions,article_total_strength,article_avg_strength
0,-9222795471790223670,26,5,6,49.0,1.884615
1,-9216926795620865886,21,10,13,33.0,1.571429
2,-9194572880052200111,29,16,18,44.0,1.517241
3,-9192549002213406534,56,45,47,65.0,1.160714
4,-9190737901804729417,9,4,5,10.0,1.111111
...,...,...,...,...,...,...
2982,9213260650272029784,11,10,10,11.0,1.000000
2983,9215261273565326920,30,12,14,39.0,1.300000
2984,9217155070834564627,16,6,7,24.0,1.500000
2985,9220445660318725468,52,27,32,54.0,1.038462


## 16. Article interaction-type counts

In [176]:
interaction_pivot = pd.crosstab(consumer["item_id"], consumer["interaction_type"]).reset_index()

interaction_pivot.columns.name = None

interaction_pivot = (interaction_pivot.rename(columns={col: f"article_{col}_count" for col in interaction_pivot.columns if col != "item_id"}))

In [177]:
article_interactions = article_interactions.merge(interaction_pivot,on="item_id",how="left")

In [178]:
article_interactions

,item_id,article_total_interactions,article_unique_users,article_unique_sessions,article_total_strength,article_avg_strength,article_content_commented_on_count,article_content_followed_count,article_content_liked_count,article_content_saved_count,article_content_watched_count
0,-9222795471790223670,26,5,6,49.0,1.884615,2,3,4,1,16
1,-9216926795620865886,21,10,13,33.0,1.571429,1,1,3,1,15
2,-9194572880052200111,29,16,18,44.0,1.517241,1,1,4,2,21
3,-9192549002213406534,56,45,47,65.0,1.160714,1,0,5,0,50
4,-9190737901804729417,9,4,5,10.0,1.111111,0,0,1,0,8
...,...,...,...,...,...,...,...,...,...,...,...
2982,9213260650272029784,11,10,10,11.0,1.000000,0,0,0,0,11
2983,9215261273565326920,30,12,14,39.0,1.300000,0,0,3,3,24
2984,9217155070834564627,16,6,7,24.0,1.500000,2,0,0,0,14
2985,9220445660318725468,52,27,32,54.0,1.038462,0,0,2,0,50


## 17. Article Engagement Rates

In [179]:
count_columns = [col for col in article_interactions.columns if col.endswith("_count")]
for col in count_columns:
    if col == "article_total_interactions":
        continue
    rate_name = (col.replace("_count", "_rate"))

    article_interactions[rate_name] = (article_interactions[col] / article_interactions["article_total_interactions"].replace(0, np.nan)).fillna(0)

## 18. Article Recency

In [180]:
# Ensure consumer event_datetime is datetime type
consumer["event_datetime"] = pd.to_datetime(consumer["event_datetime"])

# Use the maximum observed consumer interaction timestamp
reference_time = consumer["event_datetime"].max()

# Drop duplicate status columns if cell was executed previously
cols_to_drop = [c for c in ["content_event_datetime", "latest_content_status", "is_available"] if c in article_features.columns]
if cols_to_drop:
    article_features = article_features.drop(columns=cols_to_drop)

# Join latest article event timestamp
article_features = article_features.merge(content_status[["item_id","content_event_datetime","latest_content_status", "is_available"]], on="item_id",how="left")

# Convert content timestamp to datetime
article_features["content_event_datetime"] = pd.to_datetime(article_features["content_event_datetime"])

# Calculate article age in hours
article_features["article_age_hours"] = ((reference_time - article_features["content_event_datetime"]).dt.total_seconds() / 3600)

# Avoid negative values caused by inconsistent timestamps
article_features["article_age_hours"] = article_features["article_age_hours"].clip(lower=0)

# Calculate article age in days
article_features["article_age_days"] = article_features["article_age_hours"] / 24

# Exponential Frehness score
article_features["article_recency_score"] = np.exp(-RECENCY_LEMBDA * article_features["article_age_days"])

## 19. Merge Article Behavioral Features

In [181]:
article_features = article_features.merge(article_interactions, on="item_id", how="left")

article_features = (article_features.fillna(0))

In [182]:
# Restore categorical/text columns after numerical fill.
text_columns = ["title","text_description", "language", "item_type", "producer_id", "producer_country", "producer_location", "item_url", "latest_content_status"]

for col in text_columns:
    if col in article_features.columns:
        article_features[col] = (article_features[col].replace(0,"unknown").fillna("unknown"))

## 20. User-level features

In [183]:
user_features = (consumer.groupby("consumer_id").agg(
        # Total activity.
        user_total_interactions=("item_id", "size"),
        # Breadth of consumed content.
        user_unique_articles=("item_id", "nunique"),
        # Number of sessions.
        user_unique_sessions=("consumer_session_id", "nunique"),
        # Total feedback strength.
        user_total_strength=("interaction_strength", "sum"),
        # Average feedback strength.
        user_avg_strength=("interaction_strength", "mean"),
        # First activity.
        user_first_event=("event_datetime", "min"),
        # Most recent activity.
        user_last_event=("event_datetime", "max"),
        # Number of active days.
        user_active_days=("event_date", "nunique")).reset_index()
)

## 21. User Interaction Types Features

In [184]:
user_interaction_pivot = pd.crosstab(consumer["consumer_id"],consumer["interaction_type"]).reset_index()

user_interaction_pivot.columns.name = None

user_interaction_pivot = (user_interaction_pivot.rename(columns={col:f"user_{col}_count" for col in user_interaction_pivot.columns if col != "consumer_id"}))

user_features = user_features.merge(user_interaction_pivot,on="consumer_id",how="left")

## 22. User Engagement Rates

In [185]:
for col in user_features.columns:
    if (col.startswith("user_") and col.endswith("_count") and col != "user_total_interactions"):
        rate_name = (col.replace("_count", "_rate"))
        user_features[rate_name] = (user_features[col] / user_features["user_total_interactions"].replace(0, np.nan)).fillna(0)

## 23. User Recency

In [186]:
user_features["user_days_since_last_event"] = ((reference_time - user_features["user_last_event"]).dt.total_seconds() / 86400).clip(lower=0)

user_features["user_days_active_span"] = ((user_features["user_last_event"] -user_features["user_first_event"]).dt.total_seconds() / 86400).clip(lower=0)

## 24. User Activity Intensity

In [187]:
user_features["user_interactions_per_active_day"] = (user_features["user_total_interactions"] /user_features["user_active_days"].replace(0,np.nan)).fillna(0)

user_features["user_articles_per_session"] = (user_features["user_unique_articles"] /user_features["user_unique_sessions"].replace(0, np.nan)).fillna(0)

## 25. User Device Preference

In [188]:
user_device = pd.crosstab(consumer["consumer_id"], consumer["consumer_device_info"])

user_device = (user_device.div(user_device.sum(axis=1), axis=0).add_prefix("user_device_share_").reset_index())

user_features = user_features.merge(user_device,on="consumer_id",how="left")

## 26. User Country Preference

In [189]:
user_country = pd.crosstab(consumer["consumer_id"], consumer["country"])

user_country = (user_country.div(user_country.sum(axis=1), axis=0).add_prefix("user_country_share_").reset_index())

user_features = user_features.merge(user_country, on="consumer_id",how="left")

## 27. User Preferred Interaction Hour

In [190]:
user_hour = (consumer.groupby(["consumer_id", "event_hour"]).size().reset_index(
name="hour_interactions"))

user_preferred_hour = (user_hour.sort_values(["consumer_id","hour_interactions"],ascending=[True,False]).drop_duplicates(subset=["consumer_id"])[["consumer_id","event_hour"]].rename(columns={"event_hour": "user_preferred_hour"}))

user_features = user_features.merge(user_preferred_hour,on="consumer_id", how="left")

## 28. Session Features

In [191]:
session_features = (consumer.groupby(["consumer_id", "consumer_session_id"]).agg(session_events=("item_id","size"),session_unique_articles=("item_id","nunique"),session_start=("event_datetime","min"),session_end=("event_datetime","max"),session_total_strength=("interaction_strength","sum")).reset_index())

## 29. Session Duration

In [192]:
session_features["session_duration_minutes"] = ((session_features["session_end"] -session_features["session_start"]).dt.total_seconds() / 60).clip(lower=0)

session_features["session_events_per_minute"] = (session_features["session_events"] /session_features["session_duration_minutes"].replace(0,np.nan)).fillna(0)

## 30. Session Recency

In [193]:
# Define recency decay factor (e.g., 0.1 corresponds to a half-life of ~7 days)
RECENCY_LAMBDA = 0.1

session_features["session_days_since_start"] = ((reference_time -session_features["session_start"]).dt.total_seconds()/ 86400).clip(lower=0)

session_features["session_recency_score"] = np.exp(-RECENCY_LAMBDA * session_features["session_days_since_start"])

## 31. Most Recent Session Per User

In [194]:
latest_session = (session_features.sort_values(["consumer_id", "session_end"]).groupby("consumer_id").tail(1).copy())

latest_session = latest_session[["consumer_id","consumer_session_id", "session_events","session_unique_articles", "session_duration_minutes", "session_events_per_minute","session_total_strength", "session_recency_score"]]

latest_session = latest_session.rename(
    columns={col: f"current_{col}" for col in latest_session.columns if col not in ["consumer_id"]})

## 32. Merge user + Current Session Features

In [195]:
user_features = user_features.merge(latest_session, on="consumer_id", how="left")

## 33. User × Article Historical Interaction Features

In [196]:
user_item_features = (consumer.groupby(["consumer_id", "item_id"]).agg(
        # Number of interactions between user and article.
        user_item_interactions=("interaction_type", "size"),
        # Sum of interaction strengths.
        user_item_strength=("interaction_strength", "sum"),
        # Average interaction strength.
        user_item_avg_strength=("interaction_strength", "mean"),
        # First interaction.
        user_item_first_event=("event_datetime", "min"),
        # Most recent interaction.
        user_item_last_event=("event_datetime", "max"),
        # Number of different interaction types.
        user_item_interaction_types=("interaction_type", "nunique"),
        # Number of sessions involving this article.
        user_item_sessions=("consumer_session_id", "nunique")).reset_index()
)

## 34. User Article Recency

In [197]:
user_item_features["days_since_user_item_interaction"] = ((reference_time - user_item_features["user_item_last_event"]).dt.total_seconds() / 86400).clip(lower=0)

user_item_features["user_item_recency_score"] = np.exp(-RECENCY_LAMBDA * user_item_features["days_since_user_item_interaction"])

## 35. Seen Article Flag

In [198]:
user_item_features["has_seen_article"] = 1

## 36. User × Producer Affinity

In [199]:
user_producer = (consumer.merge(article_features[["item_id","producer_id"]],on="item_id",how="left").groupby(["consumer_id","producer_id"]).agg(user_producer_interactions=("item_id","size"),user_producer_strength=("interaction_strength","sum"),user_producer_articles=("item_id","nunique")).reset_index())

## 37. User Producer Affinity Score

In [200]:
user_producer["user_producer_affinity"] = (user_producer["user_producer_strength"] / user_producer["user_producer_strength"].groupby(user_producer["consumer_id"]).transform("sum").replace(0,np.nan)).fillna(0)

## 38. Article Producer Popularity

In [201]:
producer_features = (consumer.merge(article_features[["item_id", "producer_id"]], on="item_id",how="left").groupby("producer_id").agg(producer_total_interactions=("consumer_id","size"),producer_unique_users=("consumer_id","nunique"),producer_unique_articles=("item_id","nunique"),producer_total_strength=("interaction_strength","sum")).reset_index())

## 39. Producer Engagement

In [202]:
producer_features["producer_avg_strength"] = (producer_features["producer_total_strength"] / producer_features["producer_total_interactions"].replace(0,np.nan)).fillna(0)

## 40. Build Complete User-Item Feature Table

In [203]:
master_features = (user_item_features.merge(user_features, on="consumer_id", how="left").merge(article_features, on="item_id", how="left"))

In [204]:
master_features

,consumer_id,item_id,user_item_interactions,user_item_strength,user_item_avg_strength,user_item_first_event,user_item_last_event,user_item_interaction_types,user_item_sessions,days_since_user_item_interaction,...,article_content_commented_on_count,article_content_followed_count,article_content_liked_count,article_content_saved_count,article_content_watched_count,article_content_commented_on_rate,article_content_followed_rate,article_content_liked_rate,article_content_saved_rate,article_content_watched_rate
0,-9223121837663643404,-8949113594875411859,1,1.0,1.0,2016-05-05 12:42:07+00:00,2016-05-05 12:42:07+00:00,1,1,299.277593,...,3.0,0.0,7.0,2.0,43.0,0.054545,0.000000,0.127273,0.036364,0.781818
1,-9223121837663643404,-8377626164558006982,1,1.0,1.0,2016-09-15 11:25:07+00:00,2016-09-15 11:25:07+00:00,1,1,166.331065,...,4.0,5.0,4.0,3.0,37.0,0.075472,0.094340,0.075472,0.056604,0.698113
2,-9223121837663643404,-8208801367848627943,1,1.0,1.0,2016-07-28 11:51:42+00:00,2016-07-28 11:51:42+00:00,1,1,215.312604,...,8.0,8.0,25.0,8.0,217.0,0.030075,0.030075,0.093985,0.030075,0.815789
3,-9223121837663643404,-8187220755213888616,1,1.0,1.0,2016-07-06 16:51:37+00:00,2016-07-06 16:51:37+00:00,1,1,237.104329,...,0.0,0.0,2.0,1.0,32.0,0.000000,0.000000,0.057143,0.028571,0.914286
4,-9223121837663643404,-7423191370472335463,8,8.0,1.0,2016-11-09 10:26:06+00:00,2016-11-17 09:56:18+00:00,1,5,103.392743,...,1.0,1.0,5.0,1.0,76.0,0.011905,0.011905,0.059524,0.011905,0.904762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40705,9210530975708218054,8477804012624580461,4,10.0,2.5,2017-02-08 18:14:45+00:00,2017-02-08 18:15:29+00:00,4,1,20.046088,...,0.0,2.0,3.0,2.0,38.0,0.000000,0.044444,0.066667,0.044444,0.844444
40706,9210530975708218054,8526042588044002101,1,1.0,1.0,2016-12-28 01:16:00+00:00,2016-12-28 01:16:00+00:00,1,1,62.754063,...,0.0,0.0,1.0,2.0,32.0,0.000000,0.000000,0.028571,0.057143,0.914286
40707,9210530975708218054,8856169137131817223,1,1.0,1.0,2016-10-18 11:41:43+00:00,2016-10-18 11:41:43+00:00,1,1,133.319537,...,0.0,0.0,1.0,1.0,7.0,0.000000,0.000000,0.111111,0.111111,0.777778
40708,9210530975708218054,8869347744613364434,1,1.0,1.0,2016-12-09 14:49:53+00:00,2016-12-09 14:49:53+00:00,1,1,81.188866,...,0.0,0.0,3.0,4.0,70.0,0.000000,0.000000,0.038961,0.051948,0.909091


In [205]:
master_features = master_features.merge(producer_features, on="producer_id", how="left")

In [206]:
master_features = master_features.merge(user_producer[["consumer_id", "producer_id","user_producer_affinity"]], on=["consumer_id", "producer_id"], how="left")

## 41. Fill Missing Affinity

In [207]:
master_features["user_producer_affinity"] = (master_features["user_producer_affinity"].fillna(0))

## 42. User × Article Popularity Features

In [208]:
master_features["article_popularity_log"] = np.log1p(master_features["article_total_interactions"])

master_features["article_user_reach_log"] = np.log1p(master_features["article_unique_users"])

master_features["producer_popularity_log"] = np.log1p(master_features["producer_total_interactions"])

## 43. Novelty feature

In [209]:
master_features["article_novelty_score"] = (1 / (1 + np.log1p(master_features["article_total_interactions"])))

## 43. User Specific Novelty

In [210]:
# A candidate is more novel when:
#     - the user has little/no historical interaction with it
#     - it is not already heavily consumed by the user

master_features["user_article_novelty"] = (1 / (1 + master_features["user_item_interactions"]))

## 44. Recency × Popularity

In [211]:
master_features["fresh_popularity_score"] = (master_features["article_popularity_log"] * master_features["article_recency_score"])

## 45. User Activity × Article Popularity

In [212]:
master_features["article_popularity_per_user_activity"] = (master_features["article_total_interactions"] / (1 + master_features["user_total_interactions"]))

## 46. Contextual Features

In [213]:
context_features = consumer[["consumer_id", "item_id", "consumer_session_id", "event_hour","event_day_of_week", "is_weekend", "consumer_device_info", "consumer_location", "country"]].copy()

## 47. User Preferred Dominant Device

In [214]:
dominant_device = (consumer.groupby(["consumer_id", "consumer_device_info"]).size().reset_index(name="device_count").sort_values(["consumer_id", "device_count"], ascending=[True,False]).drop_duplicates(subset=["consumer_id"])[["consumer_id","consumer_device_info"]].rename(columns={"consumer_device_info": "user_dominant_device"}))

master_features = master_features.merge(dominant_device,on="consumer_id",how="left")

## 48. User Article Geographic Affinity

In [215]:
# Determine user country column name dynamically
user_country_col = next((col for col in ["user_dominant_country", "user_country", "consumer_country", "country"] if col in master_features.columns), None)

# Determine producer country column name dynamically
producer_country_col = next((col for col in ["producer_country", "item_producer_country", "producer_country_content"] if col in master_features.columns),  None)

if user_country_col and producer_country_col:
    master_features["same_country_as_producer"] = (master_features[user_country_col].fillna("unknown") == master_features[producer_country_col].fillna("unknown")).astype(int)
else:
    print(f"Warning: Could not compute geographic affinity. Missing columns in master_features.")
    print(f"Available columns matching 'country': {[c for c in master_features.columns if 'country' in c]}")
    master_features["same_country_as_producer"] = 0

Available columns matching 'country': ['user_country_share_AR', 'user_country_share_AU', 'user_country_share_BR', 'user_country_share_CA', 'user_country_share_CH', 'user_country_share_CL', 'user_country_share_CN', 'user_country_share_CO', 'user_country_share_DE', 'user_country_share_ES', 'user_country_share_GB', 'user_country_share_IE', 'user_country_share_IN', 'user_country_share_IS', 'user_country_share_IT', 'user_country_share_JP', 'user_country_share_KR', 'user_country_share_MY', 'user_country_share_NL', 'user_country_share_PT', 'user_country_share_SG', 'user_country_share_US', 'user_country_share_ZZ', 'producer_country']


## 49. Prepare categorical features

In [216]:
categorical_columns = ["language", "item_type", "producer_id", "producer_country", "producer_location", "user_dominant_device", "user_dominant_country"]

for col in categorical_columns:
    if col in master_features.columns:
        master_features[col] = (master_features[col].fillna("unknown").astype(str))

## 50. Numerical Cleanup

In [217]:
numeric_columns = (master_features.select_dtypes(include=["int64", "int32", "float64", "float32"]).columns)

master_features[numeric_columns] = (master_features[numeric_columns].replace([np.inf, -np.inf],np.nan).fillna(0))

## 51. Create Explicit Feature List

In [218]:
ID_COLUMNS = ["consumer_id", "item_id"]

NON_FEATURE_COLUMNS = ["consumer_id","item_id","user_item_first_event","user_item_last_event","user_first_event","user_last_event","content_event_datetime","title","text_description","item_url","event_datetime"]

In [219]:
# Candidate model features.
MODEL_FEATURES = [col for col in master_features.columns if col not in NON_FEATURE_COLUMNS]
print("Number of model features:",len(MODEL_FEATURES))
print("nFeatures:")
for feature in MODEL_FEATURES:
    print(feature)

Number of model features: 1200
nFeatures:
user_item_interactions
user_item_strength
user_item_avg_strength
user_item_interaction_types
user_item_sessions
days_since_user_item_interaction
user_item_recency_score
has_seen_article
user_total_interactions
user_unique_articles
user_unique_sessions
user_total_strength
user_avg_strength
user_active_days
user_content_commented_on_count
user_content_followed_count
user_content_liked_count
user_content_saved_count
user_content_watched_count
user_content_commented_on_rate
user_content_followed_rate
user_content_liked_rate
user_content_saved_rate
user_content_watched_rate
user_days_since_last_event
user_days_active_span
user_interactions_per_active_day
user_articles_per_session
user_device_share_Android - Native Mobile App
user_device_share_Mozilla/5.0 (Android 6.0.1; Mobile; rv:47.0) Gecko/47.0 Firefox/47.0
user_device_share_Mozilla/5.0 (Linux; Android 4.4.2; ASUS_T00J Build/KVT49L) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/49.0.2623.105 Mobi

## 52. Create Compact Traning Features Tables

In [220]:
feature_table = master_features[["consumer_id","item_id"] + MODEL_FEATURES].copy()
print("Feature table shape:", feature_table.shape)
display(feature_table.head())

Feature table shape: (40710, 1202)


,consumer_id,item_id,user_item_interactions,user_item_strength,user_item_avg_strength,user_item_interaction_types,user_item_sessions,days_since_user_item_interaction,user_item_recency_score,has_seen_article,...,user_producer_affinity,article_popularity_log,article_user_reach_log,producer_popularity_log,article_novelty_score,user_article_novelty,fresh_popularity_score,article_popularity_per_user_activity,user_dominant_device,same_country_as_producer
0,-9223121837663643404,-8949113594875411859,1,1.0,1.0,1,1,299.277593,1.005864e-13,1,...,0.140625,4.025352,3.713572,7.452982,0.198991,0.500000,0.000001,0.859375,Mozilla/5.0 (Windows NT 6.3; WOW64) AppleWebKi...,0
1,-9223121837663643404,-8377626164558006982,1,1.0,1.0,1,1,166.331065,5.974941e-08,1,...,0.015625,3.988984,3.135494,6.947937,0.200442,0.500000,0.000938,0.828125,Mozilla/5.0 (Windows NT 6.3; WOW64) AppleWebKi...,0
2,-9223121837663643404,-8208801367848627943,1,1.0,1.0,1,1,215.312604,4.457511e-10,1,...,0.062500,5.587249,4.983607,7.171657,0.151808,0.500000,0.000116,4.156250,Mozilla/5.0 (Windows NT 6.3; WOW64) AppleWebKi...,0
3,-9223121837663643404,-8187220755213888616,1,1.0,1.0,1,1,237.104329,5.043010e-11,1,...,0.140625,3.583519,3.091042,7.452982,0.218173,0.500000,0.000025,0.546875,Mozilla/5.0 (Windows NT 6.3; WOW64) AppleWebKi...,0
4,-9223121837663643404,-7423191370472335463,8,8.0,1.0,1,5,103.392743,3.233778e-05,1,...,0.265625,4.442651,4.043051,7.118016,0.183734,0.111111,0.016338,1.312500,Mozilla/5.0 (Windows NT 6.3; WOW64) AppleWebKi...,0


## 53. Create Article Embeddings

In [221]:
article_features["article_text"] = (article_features["title"].fillna("") +" " + article_features["text_description"].fillna(""))

## Save Individual Features Tables

In [222]:
# User features
user_features.to_parquet(FEATURE_DIR / "user_features.parquet",index=False)

# Article features
article_features.to_parquet(FEATURE_DIR / "article_features.parquet",index=False)

# Session features
session_features.to_parquet(FEATURE_DIR / "session_features.parquet",index=False)

# User-item features
user_item_features.to_parquet(FEATURE_DIR / "user_item_features_raw.parquet",index=False)

# Producer features
producer_features.to_parquet(FEATURE_DIR / "producer_features.parquet",index=False)

##  Save Features Metadata

In [223]:
feature_metadata = []
for feature in MODEL_FEATURES:
    feature_metadata.append({"feature_name": feature,"dtype": str(master_features[feature].dtype),"missing_count": int(master_features[feature].isna().sum()),"unique_values": int(master_features[feature].nunique())})

feature_metadata = pd.DataFrame(feature_metadata)

feature_metadata.to_csv(FEATURE_DIR /"feature_metadata.csv",index=False)

display(feature_metadata.head(20))

,feature_name,dtype,missing_count,unique_values
0,user_item_interactions,int64,0,38
1,user_item_strength,float64,0,50
2,user_item_avg_strength,float64,0,135
3,user_item_interaction_types,int64,0,5
4,user_item_sessions,int64,0,22
5,days_since_user_item_interaction,float64,0,40483
6,user_item_recency_score,float64,0,40483
7,has_seen_article,int64,0,1
8,user_total_interactions,int64,0,220
9,user_unique_articles,int64,0,157
